In [1]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models, metrics

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [2]:
PROCESSED_DATA_PATH = "../processed_data"
ds_train = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'train'))
ds_val = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'val'))
ds_test = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'test'))

In [3]:
base_model = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

NUM_CLASSES = 5

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', metrics.SparseTopKCategoricalAccuracy(k=1, name='mAP')]
)

In [4]:
EPOCHS = 50
print(f"🚀 Launching MobileNetV3 Large training pipeline...")
start_time = time.time()

history = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS)

end_time = time.time()
mobilenet_total_time = end_time - start_time
print(f"\nTotal MobileNetV3 Training Time: {mobilenet_total_time:.2f} seconds")

🚀 Launching MobileNetV3 Large training pipeline...
Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 204s 1s/step - accuracy: 0.3332 - loss: 1.4290 - mAP: 0.3332 - val_accuracy: 0.3346 - val_loss: 1.3724 - val_mAP: 0.3346
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.3578 - loss: 1.3642 - mAP: 0.3578 - val_accuracy: 0.3346 - val_loss: 1.3411 - val_mAP: 0.3346
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 155s 1s/step - accuracy: 0.3875 - loss: 1.3273 - mAP: 0.3875 - val_accuracy: 0.3493 - val_loss: 1.3157 - val_mAP: 0.3493
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - accuracy: 0.4091 - loss: 1.3002 - mAP: 0.4091 - val_accuracy: 0.3640 - val_loss: 1.2937 - val_mAP: 0.3640
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 120s 896ms/step - accuracy: 0.4260 - loss: 1.2829 - mAP: 0.4260 - val_accuracy: 0.4724 - val_loss: 1.2749 - val_mAP: 0.4724
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 115s 860ms/step - accuracy: 0.4328 - loss: 1.2711 - mAP: 0.4328 - val_accuracy: 0.4154 - val_loss: 1.

In [11]:
import os
import csv

# Ensure the results directory exists
os.makedirs('../results', exist_ok=True)

# 1. Capture the history dictionary from your MobileNet run
history_dict = history.history
csv_path = '../results/mobilenetv3_history.csv'

# 2. Extract metrics and write them row-by-row to CSV
with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    keys = list(history_dict.keys())
    writer.writerow(keys) # Header row
    
    num_epochs = len(history_dict[keys[0]])
    for i in range(num_epochs):
        row = [history_dict[k][i] for k in keys]
        writer.writerow(row)

# 3. Save the model configuration
model = tf.keras.models.load_model("../results/mobilenetv3_model.keras")

# 4. Log the execution time
with open('../results/mobilenetv3_time.txt', 'w') as f:
    f.write(str(mobilenet_total_time))

print("💾 SUCCESS! MobileNetV3 metrics exported to CSV and model architecture saved.")

💾 SUCCESS! MobileNetV3 metrics exported to CSV and model architecture saved.


C:\Users\HP\miniconda3\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 4 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
